# HaroCLIP — full end-to-end pipeline on Colab

Runs the whole pipeline — **ingestion → processing → highlight detection
(Claude API) → reframe → captioning** — for either:

- **a brand-new video** (`SOURCE_URL` in the config cell), or
- **an existing `ingestion_job_id`** already in the DB (`EXISTING_JOB_ID`),
  to resume a run that got interrupted partway or to redo later stages —
  every stage after ingestion is idempotent, so this safely picks up
  wherever the job actually left off (or use `FORCE = True` to redo
  everything).

This mirrors `src/pipeline/run.py`'s exact orchestration (`--url` XOR
`--job-id`), replicated as separate cells instead of shelling out to the
CLI — `main()` there calls `sys.exit(1)` on failure, which would kill a
Colab kernel, so each stage below uses plain Python control flow and prints
its own status instead.

For a narrower use case — resuming a job whose highlights are **already
selected**, without ever calling Claude again — see
`notebooks/haroclip_colab.ipynb` instead. This notebook always calls Claude
for highlight detection unless you resume a job whose highlights stage is
already `READY` (in which case that stage's own idempotency short-circuits
it, same as the CLI).

**Before running**: put `src/` under a folder in your Google Drive, e.g.
`MyDrive/HaroCLIP/src/` (an empty/fresh `data/` is fine — it's created
automatically). Set `DRIVE_ROOT` and the mode variables in the config cell.

**Run cells top to bottom, in order, once per fresh runtime.** The env-var
cell must run before any cell that imports from `src` — `src/utils/db.py`
reads `DATA_DIR` once at import time, so importing out of order binds it to
the wrong path for the rest of the session (fix: Runtime → Restart runtime,
then run in order).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config — pick exactly one of `SOURCE_URL` / `EXISTING_JOB_ID`

Must run before any `src` import (see note above).

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/HaroCLIP"  # folder containing src/

# --- Mode: set exactly one of these two, leave the other as None ---
SOURCE_URL = "https://youtu.be/xxxxxxxxxxx"  # a brand-new video
EXISTING_JOB_ID = None                        # or: resume an existing ingestion_job_id

FORCE = False  # redo every stage even if already ready (per pipeline/run.py's --force)

# --- Optional campaign context: at most ONE of the three ---
CAMPAIGN_CONTEXT = None  # inline text
CAMPAIGN_FILE = None     # path to a .txt file (read as-is)
# Must be a path Colab's VM can actually read -- not a path on your own
# computer. Easiest: drop the PDF in the same Drive folder as DRIVE_ROOT and
# point here, e.g. f"{DRIVE_ROOT}/campaign_brief.pdf". Alternative: upload it
# directly for this session only -- see the upload snippet in section 9b.
CAMPAIGN_PDF = None      # e.g. f"{DRIVE_ROOT}/campaign_brief.pdf" -- summarized via Claude Haiku

# ANTHROPIC_API_KEY and HF_TOKEN are entered via getpass in section 7 below,
# not as plain variables here -- HF_TOKEN is a credential too, so it shouldn't
# sit in the notebook file as cleartext any more than the Anthropic key should.

assert bool(SOURCE_URL) != bool(EXISTING_JOB_ID), \
    "set exactly one of SOURCE_URL or EXISTING_JOB_ID, not both/neither"
assert sum(bool(x) for x in (CAMPAIGN_CONTEXT, CAMPAIGN_FILE, CAMPAIGN_PDF)) <= 1, \
    "at most one of CAMPAIGN_CONTEXT / CAMPAIGN_FILE / CAMPAIGN_PDF"

DATA_DIR = f"{DRIVE_ROOT}/data"
os.environ["DATA_DIR"] = DATA_DIR
os.environ["HF_HUB_DISABLE_XET"] = "1"
# YOLOV8_FACE_WEIGHTS_PATH has its own hardcoded default ("data/models/...")
# independent of DATA_DIR (src/detection/face_detector.py) -- must be set
# explicitly so it resolves under the Drive-mounted data/ tree too.
os.environ["YOLOV8_FACE_WEIGHTS_PATH"] = f"{DATA_DIR}/models/yolov8x-face-lindevs.pt"

print("mode:", "fresh URL" if SOURCE_URL else f"resume job {EXISTING_JOB_ID}")
print("DATA_DIR =", os.environ["DATA_DIR"])

## 3. System dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg sqlite3

## 4. Python dependencies

Mirrors `requirements.txt`'s active deps plus the heavy ML packages that stay
commented out there (installed separately, same convention as
`scripts/entrypoint.sh`). `torch`/`torchvision` are **not** reinstalled —
Colab's runtime already ships a CUDA-matched build.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pydantic python-dotenv httpx python-multipart \
    ffmpeg-python opencv-python numpy anthropic yt-dlp sqlalchemy
!pip install -q faster-whisper ultralytics supervision python_speech_features
# ctranslate2 (faster-whisper's backend) version compatibility on Colab has
# been a moving target across test sessions -- Colab's assigned GPU host
# (and its driver/CUDA/cuDNN bundle) can differ between sessions, so no
# single pin has proven reliable yet. Left unpinned here (whatever
# faster-whisper's own dependency resolution picks) since a driver seen in
# testing (580.82.07, CUDA 13.0 -- see `!nvidia-smi`) comfortably supports
# recent ctranslate2 releases. If step 11 (Highlights) fails loading the
# whisper model, see the troubleshooting cell right below this one before
# reaching for a pin -- diagnose first, since the *direction* to pin
# (older vs newer) depends on which specific host you landed on this
# session.

If step 11 (Highlights) fails on whisper/`ctranslate2` load, the exact error
message tells you which direction to pin -- Colab's assigned GPU host
varies session to session, so there's no one fix that always works. Run
these diagnostics first:
```python
# !nvidia-smi   # look at the "CUDA Version: X.Y" in the header -- the max
#               # CUDA runtime the driver actually supports right now
# !pip show ctranslate2 | grep Version
```
Then, back in section 4's dependency cell, add ONE of these right after the
`faster-whisper` install line depending on the error you actually saw, and
re-run from there (restart runtime first -- `ctranslate2` is a native
extension, pip-reinstalling it in an already-running kernel has no effect
until the process restarts):
```python
# "CUDA driver version is insufficient for CUDA runtime version"
# -> ctranslate2 needs a newer CUDA runtime than this driver supports, pin OLDER:
# !pip install -q "ctranslate2==4.4.0"

# "Could not load library libcudnn_ops_infer.so.8" (or similar .so.8 file)
# -> ctranslate2 wants cuDNN8 but this environment only has cuDNN9, pin NEWER:
# !pip install -q "ctranslate2==4.5.0"
```
As a last resort (works, but no GPU/faster inference for transcription --
everything else in the pipeline still uses the GPU normally), force
CPU-only whisper for just this run:
```python
# import os
# os.environ["WHISPER_DEVICE"] = "cpu"
# os.environ["WHISPER_COMPUTE_TYPE"] = "int8"
```
run this before step 11's cell (must be set before `transcribe()` is
called, same ordering caveat as `DATA_DIR`).

Separately, if a cell fails on `libcublas.so.12: cannot open shared object
file`, run this (same fix as `entrypoint.sh` step 4 — usually not needed on
Colab, Colab's own torch build typically already resolves this):
```python
# import subprocess
# cublas_cudnn_path = subprocess.check_output([
#     "python3", "-c",
#     "import os, nvidia.cublas.lib, nvidia.cudnn.lib; "
#     "print(os.path.dirname(nvidia.cublas.lib.__file__) + ':' + os.path.dirname(nvidia.cudnn.lib.__file__))",
# ]).decode().strip()
# os.environ["LD_LIBRARY_PATH"] = cublas_cudnn_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
```

## 5. Put `src/` on the import path

In [ ]:
import sys

if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)  # parent of src/, so `from src.xxx import yyy` resolves

import src  # noqa: F401 -- sanity check the import path is correct
print("src imported from:", src.__file__)

## 6. YOLOv8-face weights (downloaded once, persisted on Drive)

In [ ]:
import pathlib

weights_path = pathlib.Path(os.environ["YOLOV8_FACE_WEIGHTS_PATH"])
weights_path.parent.mkdir(parents=True, exist_ok=True)
if not weights_path.exists():
    print("downloading YOLOv8-face xlarge weights...")
    !curl -L "https://github.com/lindevs/yolov8-face/releases/latest/download/yolov8x-face-lindevs.pt" -o "{weights_path}"
else:
    print("weights already present:", weights_path)

## 7. API keys — Anthropic (required) + Hugging Face (optional)

Both entered via `getpass` so neither lands in the notebook file or Colab's
execution history. `ANTHROPIC_API_KEY` is needed for real highlight
detection (and campaign-PDF summarization, if used) — `anthropic.Anthropic()`
reads this env var itself; if it's unset, highlight detection fails
immediately with a clear `AnthropicError` rather than partway through a long
run. `HF_TOKEN` is optional — only avoids faster-whisper's unauthenticated-
requests rate-limit warning; leave the prompt empty to skip it.

In [ ]:
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

HF_TOKEN = getpass.getpass("HF_TOKEN (optional, press Enter to skip): ") or None
print("HF_TOKEN set:", bool(HF_TOKEN))

## 8. Open the DB

In [ ]:
from src.utils.db import DATA_DIR, SessionLocal, init_db

# Import every model module BEFORE init_db() -- Base.metadata.create_all()
# only creates tables for models that have actually been imported/registered
# with Base by the time it runs. src/pipeline/run.py gets this for free
# because it imports every service module (which transitively import their
# models) at the top of the file, before main() ever calls init_db(); here
# the imports are split across later cells, so they're pulled forward
# explicitly to avoid ending up with a DB file missing tables.
from src.ingestion.models import IngestionJob  # noqa: F401
from src.processing.models import ProcessingJob  # noqa: F401
from src.highlights.models import HighlightClip, HighlightJob  # noqa: F401
from src.reframe.models import ReframeJob  # noqa: F401
from src.captioning.models import CaptionJob  # noqa: F401

print("resolved DATA_DIR:", DATA_DIR)
assert str(DATA_DIR) == os.environ["DATA_DIR"], (
    "DATA_DIR was already bound before this cell ran -- restart the runtime "
    "and run cells in order starting from the config cell."
)

init_db()
db = SessionLocal()

## 9. Stage 1/5 — Ingestion

Fresh-URL branch validates the link (ffprobe/yt-dlp, network only) and
creates a new job; resume branch looks up the existing job and, if it's
not `READY`, stops here rather than proceeding (mirrors
`pipeline/run.py`'s exact behavior — a not-ready ingestion job blocks every
later stage regardless of `FORCE`).

In [ ]:
from pathlib import Path

from src.ingestion.enums import JobStatus
from src.ingestion.models import IngestionJob
from src.ingestion.service import create_job, run_validation

campaign_context = CAMPAIGN_CONTEXT
if CAMPAIGN_FILE:
    campaign_context = Path(CAMPAIGN_FILE).read_text(encoding="utf-8").strip()

if SOURCE_URL:
    print(f"[1/5] Ingestion: submitting {SOURCE_URL}")
    job = create_job(db, SOURCE_URL, campaign_context=campaign_context, hf_token=HF_TOKEN)
    run_validation(job.id)  # opens its own DB session internally and commits
    db.refresh(job)
    if job.status != JobStatus.READY:
        raise RuntimeError(
            f"ingestion failed: [{job.error_stage}] {job.error_message} "
            f"(job id: {job.id} -- set EXISTING_JOB_ID to this and re-run once fixed)"
        )
    print(f"[1/5] Ingestion OK: job_id={job.id} duration={job.duration_seconds}s {job.width}x{job.height}")
else:
    job = db.get(IngestionJob, EXISTING_JOB_ID)
    if job is None:
        raise RuntimeError(f"no ingestion job found for id {EXISTING_JOB_ID}")
    if job.status != JobStatus.READY:
        raise RuntimeError(f"ingestion job {job.id} is not ready (status={job.status.value})")
    if campaign_context is not None:
        job.campaign_context = campaign_context
    if HF_TOKEN is not None:
        job.hf_token = HF_TOKEN
    db.commit()
    print(f"[1/5] Ingestion: resuming from existing job_id={job.id}")

## 9b. Optional — campaign brief PDF

Only runs if `CAMPAIGN_PDF` was set in the config cell. Skip this cell
otherwise.

`CAMPAIGN_PDF` must point to a file Colab's VM can read, not a path on your
own computer. Easiest is placing the PDF in Drive (see the config cell's
comment). If you'd rather upload it just for this session instead, run this
before the cell below and it'll set `CAMPAIGN_PDF` for you:
```python
# from google.colab import files
# uploaded = files.upload()  # pick a PDF from your computer
# CAMPAIGN_PDF = "/content/" + list(uploaded.keys())[0]
```

In [ ]:
from src.campaign_brief.service import apply_campaign_brief
from src.utils.logging import get_job_logger

if CAMPAIGN_PDF:
    pdf_path = Path(CAMPAIGN_PDF)
    print(f"applying campaign brief: {pdf_path}")
    job = apply_campaign_brief(
        db, job.id, pdf_path.read_bytes(), pdf_path.name,
        logger=get_job_logger("campaign_brief", job.id),
    )
    print(f"campaign brief summarized ({len(job.campaign_context)} chars)")
else:
    print("CAMPAIGN_PDF not set, skipping")

## 10. Stage 2/5 — Processing (download + extract audio)

In [ ]:
from src.processing.enums import ProcessingStatus
from src.processing.service import run_processing

print("[2/5] Processing: downloading + extracting audio")
proc = run_processing(db, job.id, force=FORCE)
if proc.status != ProcessingStatus.READY:
    raise RuntimeError(f"processing failed: [{proc.error_stage}] {proc.error_message}")
print(f"[2/5] Processing OK: video={proc.video_path} audio={proc.audio_path}")

## 11. Stage 3/5 — Highlights (transcribe + Claude + render)

This is the only stage that calls the Claude API — skipped automatically if
resuming a job whose highlights are already `READY` and `FORCE` is `False`.

In [ ]:
from src.highlights.enums import HighlightStatus
from src.highlights.models import HighlightClip
from src.highlights.service import run_highlight_detection

print("[3/5] Highlights: transcribing + detecting + rendering")
hjob = run_highlight_detection(db, job.id, force=FORCE)
if hjob.status != HighlightStatus.READY:
    raise RuntimeError(f"highlights failed: [{hjob.error_stage}] {hjob.error_message}")

clips = (
    db.query(HighlightClip)
    .filter_by(highlight_job_id=hjob.id)
    .order_by(HighlightClip.rank)
    .all()
)
print(f"[3/5] Highlights OK: {len(clips)} clips")

## 12. Stage 4/5 — Reframe (dynamic vertical crop per clip)

In [ ]:
from src.reframe.enums import ReframeStatus
from src.reframe.service import run_reframe

print("[4/5] Reframe: dynamic vertical-crop per clip")
reframe_ready = {}
for clip in clips:
    rjob = run_reframe(db, clip.id, force=FORCE)
    reframe_ready[clip.id] = rjob.status == ReframeStatus.READY
    if rjob.status == ReframeStatus.READY:
        print(f"  clip #{clip.rank}: OK -> {rjob.output_path}")
    else:
        print(f"  clip #{clip.rank}: FAILED [{rjob.error_stage}] {rjob.error_message}")

## 13. Stage 5/5 — Captioning

In [ ]:
from src.captioning.enums import CaptionStatus
from src.captioning.service import run_captioning

print("[5/5] Captioning: burning word-burst captions per clip")
for clip in clips:
    if not reframe_ready[clip.id]:
        print(f"  clip #{clip.rank}: SKIPPED (reframe not ready)")
        continue
    cjob = run_captioning(db, clip.id, force=FORCE)
    if cjob.status == CaptionStatus.READY:
        print(f"  clip #{clip.rank}: OK -> {cjob.output_path}")
    else:
        print(f"  clip #{clip.rank}: FAILED [{cjob.error_stage}] {cjob.error_message}")

## 14. Done — verify

In [ ]:
print(f"=== DONE. ingestion job id: {job.id} ===")
print(f"logs:                        {DATA_DIR}/logs/{job.id}/")
print(f"reframed clips:               {DATA_DIR}/reframed/{job.id}/")
print(f"captioned clips (final):      {DATA_DIR}/captioned/{job.id}/")

captioned_dir = pathlib.Path(DATA_DIR) / "captioned" / job.id
outputs = sorted(captioned_dir.glob("*.mp4"))
for f in outputs:
    print(" ", f)

In [ ]:
from IPython.display import Video

# Adjust the index to preview a different clip.
Video(str(outputs[0]), embed=True, width=360) if outputs else print("no outputs yet")